In [2]:
import pandas as pd
import pickle
from scipy.signal import butter, lfilter
import numpy as np
import glob
import os
from pathlib import Path
from brainflow.data_filter import DataFilter, WindowOperations

# Parameters

In [3]:
windowsize = 10
basalscene = 0
targetscene = 3
subjects = [1, 2, 3, 4, 6, 8, 9, 10]
# Reftype changes the reference for normalization. 1 = Baseline normalization, 2 = Stimulus scene normalization (equivalent to z-score normalization),
# 3 = sliding window normalization 
reftype = 1
# Select option for regression models (input 1 or 2). Option 1 encompasses 4 feature valence model, 3 feature arousal model and 3 feature dominance model.
# Option 2 stands for 4 feature valence model, 4 feature arousal model and 3 feature dominance model.
model_option = 2

In [4]:
#base_dir = os.path.dirname(__file__)
base_dir = Path().resolve()

### Model import

In [5]:
if model_option == 1:
    v_n = 4
    a_n = 3
    d_n = 3
elif model_option == 2:
    v_n = 4
    a_n = 4
    d_n = 3

path_models = os.path.join(base_dir, 'Models')

#Val_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_val_model2_{}s.pkl'.format(windowsize)), 'rb'))
#Aro_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_aro_model2_{}s.pkl'.format(windowsize)), 'rb'))
#Dom_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_dom_model2_{}s.pkl'.format(windowsize)), 'rb'))

Val_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_val_model2_{}s_{}f.pkl'.format(windowsize,v_n)), 'rb'))
Aro_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_aro_model2_{}s_{}f.pkl'.format(windowsize,a_n)), 'rb'))
Dom_Pkl_linear = pickle.load(open(os.path.join(path_models, 'reg_dom_model2_{}s_{}f.pkl'.format(windowsize,d_n)), 'rb'))

# Functions

### Miscelaneous functions

In [6]:
# Use for resampling when needed
def resampling(signal, input_fs, output_fs):
    ratio = output_fs / input_fs

    # New length of sample
    n = round(len(signal) * ratio)

    # Perform linear interpolation
    resampled_signal = np.interp(
        np.linspace(0.0, 1.0, n, endpoint=False),  # Where to interpret
        np.linspace(0.0, 1.0, len(signal), endpoint=False),  # Known positions
        signal,
    )
    return resampled_signal


# Bandpass butterworth filter
def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype="band")
    y = lfilter(b, a, data)
    return y


# Bandpower calculation using brainflow
def brainflow_bandpowers(signal, fs, nfft):
    # Frequency band definition
    bands = [0.5, 4, 8, 12, 30, 45]

    # Compute the PSD for each frequency band
    psd = DataFilter.get_psd_welch(
        signal,
        nfft=nfft,
        overlap=nfft // 2,
        sampling_rate=fs,
        window=WindowOperations.BLACKMAN_HARRIS,
    )

    band_powers = []
    for i in range(len(bands) - 1):
        low, high = bands[i], bands[i + 1]
        power = DataFilter.get_band_power(psd, low, high)
        band_powers.append(power)

    return band_powers

### Cubic model functions (emotion and fear)

In [7]:
emotions = {
    "[-1, -1, -1]": "Sadness",
    "[-1, -1, 0]": "Sadness",
    "[-1, -1, 1]": "Desire",
    "[-1, 0, -1]": "Sadness",
    "[-1, 0, 0]": "Sadness",
    "[-1, 0, 1]": "Desire",
    "[-1, 1, -1]": "Sadness",
    "[-1, 1, 0]": "Sadness",
    "[-1, 1, 1]": "Desire",
    "[0, -1, -1]": "Hate",
    "[0, -1, 0]": "Love",
    "[0, -1, 1]": "Love",
    "[0, 0, -1]": "Sadness",
    "[0, 0, 0]": "Admiration",
    "[0, 0, 1]": "Desire",
    "[0, 1, -1]": "Sadness",
    "[0, 1, 0]": "Admiration",
    "[0, 1, 1]": "Joy",
    "[1, -1, -1]": "Hate",
    "[1, -1, 0]": "Admiration",
    "[1, -1, 1]": "Love",
    "[1, 0, -1]": "Hate",
    "[1, 0, 0]": "Admiration",
    "[1, 0, 1]": "Joy",
    "[1, 1, -1]": "Admiration",
    "[1, 1, 0]": "Admiration",
    "[1, 1, 1]": "Joy"
}

fear_emotions = {
    "[-1, -1, -1]": "Medium Fear",
    "[-1, -1, 0]": "Medium Fear",
    "[-1, -1, 1]": "Low Fear",
    "[-1, 0, -1]": "Medium Fear",
    "[-1, 0, 0]": "Low Fear",
    "[-1, 0, 1]": "No Fear",
    "[-1, 1, -1]": "Low Fear",
    "[-1, 1, 0]": "No Fear",
    "[-1, 1, 1]": "No Fear",
    "[0, -1, -1]": "High Fear",
    "[0, -1, 0]": "Medium Fear",
    "[0, -1, 1]": "Medium Fear",
    "[0, 0, -1]": "Medium Fear",
    "[0, 0, 0]": "Medium Fear",
    "[0, 0, 1]": "Low Fear",
    "[0, 1, -1]": "Medium Fear",
    "[0, 1, 0]": "Low Fear",
    "[0, 1, 1]": "No Fear",
    "[1, -1, -1]": "High Fear",
    "[1, -1, 0]": "High Fear",
    "[1, -1, 1]": "Medium Fear",
    "[1, 0, -1]": "High Fear",
    "[1, 0, 0]": "Medium Fear",
    "[1, 0, 1]": "Low Fear",
    "[1, 1, -1]": "Medium Fear",
    "[1, 1, 0]": "Low Fear",
    "[1, 1, 1]": "Low Fear"
}

# Categorization of vad values
def categorize(value):
    if value >= 1 and value < 3:
        category = -1
    elif value >= 3 and value < 6:
        category = 0
    else:
        category = 1

    return category

# Emotion and fear classification from categorical vad values
def emocion(aro, domi, val):
    # Create the key and ensure it's formatted as a string
    key = str([aro, domi, val])
    emotion_value = emotions.get(key, 'Unknown')
    fear_value = fear_emotions.get(key, 'Unknown')
    return emotion_value, fear_value


### Spheric model functions (emotion and fear)

In [8]:
## Emotion sphere ##
Descartes_Passions = {
    'Desire': np.array([1,0,0]),
    'Admiration': np.array([0,0,1]),
    'Joy': np.array([1,0,1]),
    'Love': np.array([1,-1,1]),
    'Hate': np.array([-1,-1,1]),
    'Sadness': np.array([-1, 0, -1])
}
DP_NORM = {emotion: vec / np.linalg.norm(vec) for emotion, vec in Descartes_Passions.items()}

# Emotion identification from regression values
def get_emotion_sphere(value):
    """
    Identify the closest matching emotion vector to the given input.

    Input:
    - value (np.array): The input vector representing a detected value for emotion analysis.
        where
        value[0]: valence [1,9], 
        value[1]: dominance [1,9],
        value[2]: arousal [1,9]

    Returns:
    - final_emotion (str): The emotion label with the highest similarity to the normalized input.
    - temp (float): The highest similarity score found.
    - random_emotion (np.array): The normalized version of the input vector on the unit sphere.
    """

    # Map or adjust the input value to [-1,1].
    detected_value = map_value(value)

    # Normalize the mapped detected_value vector to unit length to ensure it lies on the unit sphere.
    random_emotion = detected_value / np.linalg.norm(detected_value)

    # Initialize a temporary variable to store the highest similarity score found.
    temp = 0
    
    # Iterate over each emotion and its normalized vector in DP_NORM.
    for emotion, vec in DP_NORM.items():
        
        # Calculate the cosine similarity between random_emotion and the current emotion vector.
        # Adding 1 and dividing by 2 scales the similarity from [-1, 1] to [0, 1].
        dot = (np.dot(random_emotion, vec) + 1) / 2
        
        # Update temp and final_emotion if the current similarity score is the highest encountered.
        if dot > temp:
            temp, final_emotion = dot, emotion

    # Return the final emotion label with the highest similarity score, the score itself,
    # and the normalized random_emotion vector.
    return final_emotion, temp, random_emotion

## Fear sphere ##
def map_value(val):
    return (val - 1) / (4) - 1

def get_fear_sphere(value):
     # Define angles for rotation in radians
    theta = 2.356194490192345  # Rotation angle around the z-axis
    phi = -0.9553166181245093  # Rotation angle around the x-axis

    r= np.array([
        [np.cos(theta) * np.cos(phi), -np.sin(theta) * np.cos(phi), np.sin(phi)],
        [np.sin(theta), np.cos(theta), 0],
        [-np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)]
    ])


    # Apply the rotation matrix to the mapped vector
    # Map or adjust the input value to [-1,1].
    n_vec = np.dot(r, map_value(value))

    # Decompose the rotated vector into components
    valence, dominance, arousal = n_vec

    # Compute the polar angle phi (angle from the z-axis)
    phi = np.arctan2(np.sqrt(valence ** 2 + dominance ** 2), arousal)

    # Compute the azimuthal angle theta (angle in the xy-plane)
    theta = np.arctan2(valence, dominance)

    fear_metric= np.abs(phi - np.pi) / np.pi, theta

    return fear_metric


def classify_fearm_metric(fear_metric):
    bins= [0, 0.25, 0.50, 0.75, 1.0]
    labels = ["No Fear", "Low Fear", "Medium Fear", "High Fear"]
    index = np.digitize(fear_metric, bins, right=True) - 1
    return labels[index]

# Data import function

In [9]:
# folderpath = (
#     r"C:\Users\josee\OneDrive\Desktop\Projects\NeuroH\Coding\Fear_analysis\Data"
# )
folderpath = os.path.join(base_dir, 'RawData')

def dataimport(subjects, folderpath, scene):
    rawdata_list = []

    for subject in subjects:
        subject_str = f"S{subject}" if subject == 10 else f"S{subject:03d}"
        pattern = os.path.join(
            folderpath,
            rf"S{subject}\EEG\{subject_str}R00{scene}*\{subject_str}R00{scene}_Complete.csv",
        )
        matched_files = glob.glob(pattern)
        if len(matched_files) == 1:
            holder = pd.read_csv(matched_files[0])[
                [
                    "Timestamp",
                    "Channel_1",
                    "Channel_2",
                    "Channel_3",
                    "Channel_4",
                    "Channel_5",
                    "Channel_6",
                    "Channel_7",
                    "Channel_8",
                ]
            ].dropna()
            holder.insert(loc=0, column="Subject", value=[subject] * len(holder))
        else:

            holder = pd.read_csv(matched_files[1])[
                [
                    "Timestamp",
                    "Channel_1",
                    "Channel_2",
                    "Channel_3",
                    "Channel_4",
                    "Channel_5",
                    "Channel_6",
                    "Channel_7",
                    "Channel_8",
                ]
            ].dropna()
            holder.insert(loc=0, column="Subject", value=[subject] * len(holder))

        rawdata_list.append(holder)

    rawdata_df = pd.concat(rawdata_list, ignore_index=True)

    return rawdata_df


basaldata = dataimport(subjects, folderpath, scene=basalscene)
tscenedata = dataimport(subjects, folderpath, scene=targetscene)

# Pre-pro, processing and normalization functions

### Processing 

In [10]:
lowcut = 4.0  # Lower cutoff frequency in Hz
highcut = 45  # Upper cutoff frequency in Hz
fs = 250  # Sampling rate in Hz
wsz = windowsize * fs


def processing(data, lowcut, highcut, fs, wsz):
    featmat_list = []
    for subject in subjects:
        dataholder = data[data["Subject"] == subject][
            [
                "Channel_1",
                "Channel_2",
                "Channel_3",
                "Channel_4",
                "Channel_5",
                "Channel_6",
                "Channel_7",
                "Channel_8",
            ]
        ]

        # Apply the bandpass filter to each column
        filtered_df = dataholder.apply(
            lambda col: butter_bandpass_filter(col, lowcut, highcut, fs)
        )

        average_reference = filtered_df.mean(axis=1)
        df_average_reference = filtered_df.sub(average_reference, axis=0)
        t = np.arange(0, len(df_average_reference), 1) * 1 / fs

        # Create an empty DataFrame to store the PSD results
        feat_list = []
        num_segments = filtered_df.shape[0] // wsz
        for n in range(num_segments):
            timepoint = t[n * wsz : (n + 1) * wsz][0]
            df_temp = df_average_reference.iloc[n * wsz : (n + 1) * wsz, :]
            # Iterate over each column in your DataFrame
            psd_df = pd.DataFrame()

            for column in df_average_reference.columns:
                signal = np.ascontiguousarray(df_temp[column].values.astype(np.float64))
                # Compute the PSD for the column data and frequency bands
                psd_bands = brainflow_bandpowers(signal=signal, fs=fs, nfft=256)

                # Add the PSD values to the DataFrame
                psd_df = pd.concat(
                    [psd_df, pd.DataFrame([psd_bands])], ignore_index=True
                )
            psd_df.columns = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]

            df_t = psd_df.transpose()
            df_t.columns = ["Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2"]

            df_t = df_t.reset_index()

            # Use the melt function to reshape the DataFrame
            melted_df = pd.melt(
                df_t, id_vars="index", var_name="channel", value_name="value"
            )

            # Convert channel numbers to strings
            melted_df["channel"] = melted_df["channel"].astype(str)

            # Create a new 'channel_band' column by combining 'channel' and 'index' columns
            melted_df["channel_band"] = melted_df["channel"] + "_" + melted_df["index"]

            # Pivot the DataFrame to get the desired format
            new_df = melted_df.pivot(
                index="index", columns="channel_band", values="value"
            )

            series = new_df.stack()

            # Convert the Series back to a DataFrame with a single row
            filter_df = pd.DataFrame(series)

            valo = filter_df[0]
            valores = valo.reset_index(drop=True)
            df_modelo = pd.DataFrame(valores).transpose()

            df_modelo.columns = [
                "Fp1_Delta",
                "Fp1_Theta",
                "Fp1_Alpha",
                "Fp1_Beta",
                "Fp1_Gamma",
                "Fp2_Delta",
                "Fp2_Theta",
                "Fp2_Alpha",
                "Fp2_Beta",
                "Fp2_Gamma",
                "C3_Delta",
                "C3_Theta",
                "C3_Alpha",
                "C3_Beta",
                "C3_Gamma",
                "C4_Delta",
                "C4_Theta",
                "C4_Alpha",
                "C4_Beta",
                "C4_Gamma",
                "P7_Delta",
                "P7_Theta",
                "P7_Alpha",
                "P7_Beta",
                "P7_Gamma",
                "P8_Delta",
                "P8_Theta",
                "P8_Alpha",
                "P8_Beta",
                "P8_Gamma",
                "O1_Delta",
                "O1_Theta",
                "O1_Alpha",
                "O1_Beta",
                "O1_Gamma",
                "O2_Delta",
                "O2_Theta",
                "O2_Alpha",
                "O2_Beta",
                "O2_Gamma",
            ]

            df_pred = df_modelo.reset_index(drop=True)

            CANALES = ["Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2"]

            for channel in CANALES:
                df_pred[f"{channel}_Engagement"] = df_pred[f"{channel}_Beta"] / (
                    df_pred[f"{channel}_Theta"] + df_pred[f"{channel}_Alpha"]
                )

            for channel in CANALES:
                df_pred[f"{channel}_Fatigue"] = (
                    df_pred[f"{channel}_Alpha"] / df_pred[f"{channel}_Theta"]
                )

            for channel in CANALES:
                df_pred[f"{channel}_Excitement"] = (
                    df_pred[f"{channel}_Beta"] / df_pred[f"{channel}_Alpha"]
                )

            for channel in CANALES:
                df_pred[f"{channel}_Relaxation"] = (
                    df_pred[f"{channel}_Theta"] / df_pred[f"{channel}_Delta"]
                )

            df_pred.insert(0, "Time", [timepoint])
            feat_list.append(df_pred)
        feat_df = pd.concat(feat_list, ignore_index=True)
        feat_df.insert(loc=0, column="Subject", value=[subject] * len(feat_df))
        featmat_list.append(feat_df)
    featmat_df = pd.concat(featmat_list, ignore_index=True)
    return featmat_df


basalprocessed = processing(
    data=basaldata, fs=fs, lowcut=lowcut, highcut=highcut, wsz=wsz
)
tsceneprocessed = processing(
    data=tscenedata, fs=fs, lowcut=lowcut, highcut=highcut, wsz=wsz
)


### Normalization

In [11]:
# This line is redundant, eliminate and substitute application on this cell based on preference.
channel_cols = [
    col for col in tsceneprocessed.columns if col not in ["Subject", "Time"]
]

# Establish the size of sliding window (in seconds) for 3rd normalization alternative, change only if needed.
slidingwindowsize = 60

normdata_list = []

# Data is normalized per subject
for subject in subjects:
    data = tsceneprocessed[tsceneprocessed["Subject"] == subject].copy()
    # Baseline normalization
    if reftype == 1:
        ref_data = basalprocessed[basalprocessed["Subject"] == subject].drop(
            ["Subject", "Time"], axis=1
        )
        ref_mean = ref_data.mean()
        ref_std = ref_data.std()
        data[channel_cols] = (data[channel_cols] - ref_mean) / ref_std
    # Stimulus scene normalization
    elif reftype == 2:
        ref_data = basalprocessed[basalprocessed["Subject"] == subject].drop(
            ["Subject", "Time"], axis=1
        )
        ref_mean = ref_data.mean()
        ref_std = ref_data.std()
        data[channel_cols] = (data[channel_cols] - ref_mean) / ref_std
    # Sliding window normalization
    elif reftype == 3:
        # 
        basal = basalprocessed[basalprocessed["Subject"] == subject][-slidingwindowsize/windowsize]
        stimuli = tsceneprocessed[tsceneprocessed["Subject"] == subject]
        ref = pd.concat([basal, stimuli], ignore_index=True).drop(
            ["Subject", "Time"], axis=1
        )
        data = data.reset_index(drop=True)
        for window in range(len(data)):
            ref_mean = ref.iloc[window : window + 6].mean()
            ref_std = ref.iloc[window : window + 6].std()
            data.loc[window, channel_cols] = (
                data.loc[window, channel_cols] - ref_mean
            ) / ref_std

    normdata_list.append(data)
normdata_df = pd.concat(normdata_list, ignore_index=True)

# Model implementation

### Regression 

In [12]:
# modelinput = normdata_df[['Fp2_Theta','P7_Theta','C3_Gamma','O1_Gamma','C4_Gamma']]

# Normdata is put into different datasets to be introduced into the regression models, considering the expected features for each. Currently hardcoded
# but could be easily changed if needed.
if model_option == 1:
    # Valence 4, Arousal 3, Dominance 3
    valinput = normdata_df[['P7_Theta','C3_Gamma','O2_Gamma', 'P8_Beta']]
    aroinput = normdata_df[['Fp2_Theta','C3_Gamma','O1_Gamma']]
    dominput = normdata_df[['P7_Theta','Fp1_Gamma','O1_Beta']]
elif model_option == 2:
    # Valence 4, Arousal 4, Dominance 3
    valinput = normdata_df[['P7_Theta','C3_Gamma','O2_Gamma', 'P8_Beta']]
    aroinput = normdata_df[['Fp2_Theta','C3_Gamma','O1_Gamma','O2_Gamma']]
    dominput = normdata_df[['P7_Theta','Fp1_Gamma','O1_Beta']]

In [13]:
# valen, arous, domin = (
#     Val_Pkl_linear.predict(modelinput.values).reshape(-1),
#     Aro_Pkl_linear.predict(modelinput.values).reshape(-1),
#     Dom_Pkl_linear.predict(modelinput.values).reshape(-1),
# )
# Regression models implementation, outputs are lists
valen, arous, domin = (
    Val_Pkl_linear.predict(valinput.values).reshape(-1),
    Aro_Pkl_linear.predict(aroinput.values).reshape(-1),
    Dom_Pkl_linear.predict(dominput.values).reshape(-1),
)

c:\Users\josee\OneDrive\Documentos\GitHub\neurohumanities-lab\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\josee\OneDrive\Documentos\GitHub\neurohumanities-lab\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\josee\OneDrive\Documentos\GitHub\neurohumanities-lab\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but HistGradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [14]:
# Data clipping of vad lists, considering minimum value = 1 and maximum value = 9
valen = [min(max(x, 1), 9) for x in valen]
arous = [min(max(x, 1), 9) for x in arous]
domin = [min(max(x, 1), 9) for x in domin]

### Regression data categorization

In [15]:
# vad values are categorized using the corresponding function, results are stored in lists.
valen_categorized = list(map(categorize, valen))
arous_categorized = list(map(categorize, arous))
domin_categorized = list(map(categorize, domin))

## Cube model

In [16]:
emotion_cube = [
    emocion(arous_categorized[i], domin_categorized[i], valen_categorized[i])
    for i in range(len(arous_categorized))
]
emotion_cube = np.array(emotion_cube)

## Spheric model

In [17]:
vda = np.column_stack((valen, domin, arous))
emotion_spheric = [get_emotion_sphere(vda[i, :])[1::-1] for i in range(vda.shape[0])]
emotion_spheric = [get_emotion_sphere(vda[i, :])[1::-1] for i in range(vda.shape[0])]
detected_values = np.array([valen, domin, arous]).T
results = []
for row in detected_values:
    fear_metric, theta = get_fear_sphere(row)
    fear_label = classify_fearm_metric(fear_metric)
    results.append((fear_metric, fear_label))
fear_label_spheric = np.array(results)

### Data organization and export

In [18]:
vad_mat_arr = np.column_stack(
    [
        normdata_df["Subject"].values,
        normdata_df["Time"].values,
        valen,
        arous,
        domin,
        emotion_spheric,
        fear_label_spheric,
        emotion_cube,
    ]
)

vad_mat_df = pd.DataFrame(
    vad_mat_arr,
    columns=[
        "Subject",
        "Time",
        "Valence",
        "Arousal",
        "Dominance",
        "emo_similarity",
        "emo_spheric",
        "fear_metric",
        "fear_spheric",
        "emo_cubic",
        "fear_cubic",
    ],
)


In [19]:
vad_mat_df

,Subject,Time,Valence,Arousal,Dominance,emo_similarity,emo_spheric,fear_metric,fear_spheric,emo_cubic,fear_cubic
0,1,0.0,5.198358007348961,6.640389431425153,6.246460499527157,0.8962761686455412,Admiration,0.5173736245066963,Medium Fear,Admiration,Low Fear
1,1,10.0,5.198358007348961,6.231322053989099,6.272704265351403,0.8455022809290493,Admiration,0.4752498775443541,Low Fear,Admiration,Low Fear
2,1,20.0,5.153622724714609,6.352030362977638,6.272704265351403,0.862832753533405,Admiration,0.49267096142733513,Low Fear,Admiration,Low Fear
3,1,30.0,5.153622724714609,6.352030362977638,6.272704265351403,0.862832753533405,Admiration,0.49267096142733513,Low Fear,Admiration,Low Fear
4,1,40.0,5.153622724714609,6.352030362977638,6.272704265351403,0.862832753533405,Admiration,0.49267096142733513,Low Fear,Admiration,Low Fear
...,...,...,...,...,...,...,...,...,...,...,...
133,10,100.0,5.121891264525864,7.04377654475437,6.16081379192425,0.934183463280859,Admiration,0.5597778935549836,Medium Fear,Admiration,Low Fear
134,10,110.0,5.125891264525864,7.238107771374396,6.16081379192425,0.943299575679009,Admiration,0.5698213841516038,Medium Fear,Admiration,Low Fear
135,10,120.0,5.729858065548201,4.892336875511588,5.525244237968321,0.9029560001062047,Desire,0.16490473862344796,No Fear,Admiration,Medium Fear
136,10,130.0,5.121891264525864,7.04377654475437,5.626192896213061,0.9772887795602494,Admiration,0.6136135182535077,Medium Fear,Admiration,Medium Fear


In [20]:
normdata_df

,Subject,Time,Fp1_Delta,Fp1_Theta,Fp1_Alpha,Fp1_Beta,Fp1_Gamma,Fp2_Delta,Fp2_Theta,Fp2_Alpha,...,O1_Excitement,O2_Excitement,Fp1_Relaxation,Fp2_Relaxation,C3_Relaxation,C4_Relaxation,P7_Relaxation,P8_Relaxation,O1_Relaxation,O2_Relaxation
0,1,0.0,-0.593464,-0.596248,-0.611590,-0.614584,-0.453117,-0.447916,-0.416774,-0.615859,...,5.421938,11.160822,-0.084460,5.188957,1.428337,15.138960,10.699373,0.400493,-1.027136,-1.200153
1,1,10.0,-0.594129,-0.598146,-0.610283,-0.613487,-0.453117,-0.447963,-0.422160,-0.613378,...,-1.354527,1.713103,-3.470991,2.580690,1.037970,-0.937308,7.854442,3.518750,-1.480020,-1.119902
2,1,20.0,-0.595999,-0.599355,-0.614321,-0.618319,-0.453293,-0.448461,-0.427121,-0.617890,...,-3.081409,-0.910336,-2.933525,0.327722,1.678696,-0.893654,15.651388,4.463409,-0.697432,-1.069702
3,1,30.0,-0.596527,-0.600095,-0.613846,-0.617729,-0.453344,-0.449033,-0.428403,-0.617915,...,-0.401696,-1.193498,-5.029827,-0.217949,1.249463,-0.851583,17.879914,11.286971,-0.467379,-1.153562
4,1,40.0,-0.597260,-0.600066,-0.615916,-0.620708,-0.453336,-0.449952,-0.428625,-0.618894,...,-1.346585,-1.065626,-0.614909,0.181936,0.933567,-0.883805,11.740003,4.288812,-0.103901,-1.076878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,10,100.0,-0.399356,-0.384806,-0.376942,-0.408920,-0.419020,-0.403848,-0.412905,-0.420879,...,8.598933,7.438101,1.742460,3.112476,-1.217793,-0.986994,-1.548917,6.819620,0.561244,-0.883243
134,10,110.0,-0.399871,-0.352632,-0.393355,-0.423559,-0.416386,-0.403784,-0.416559,-0.407660,...,7.234020,7.461178,6.979369,2.559124,-0.971421,-0.494614,-1.623945,1.869939,1.222443,-0.656949
135,10,120.0,3.804081,8.324132,1.881709,5.196833,4.749612,2.563919,3.896552,4.079944,...,4.719618,-0.801020,4.354542,-0.877699,3.540764,-0.444932,-0.234485,-1.906597,2.898242,1.088449
136,10,130.0,-0.401996,-0.003245,-0.380743,-0.445306,-0.412988,-0.404460,-0.408639,-0.420523,...,84.542272,13.468330,67.716783,6.462512,-1.252361,-0.909229,-1.678595,2.025805,0.798878,-0.863912


In [21]:
outpath = os.path.join(base_dir, 'Outputs', rf'scene{targetscene}', rf'models_option_{model_option}')
# normdata_df.to_csv(os.path.join(outpath, rf'featmat_scene{targetscene}_basalnorm_{windowsize}s_v{v_n}a{a_n}d{d_n}.csv'), index=False)
# vad_mat_df.to_csv(os.path.join(outpath, rf'vadmat_scene{targetscene}_basalnorm_{windowsize}s_v{v_n}a{a_n}d{d_n}.csv'), index=False)

# normdata_df.to_csv(r'Outputs\SceneOrganized\Newnorms\featmat_scene{}_slidingnorm_{}s_v{}a{}d{}.csv'.format(targetscene, windowsize,v_n,a_n,d_n), index=False)
# vad_mat_df.to_csv(r'Outputs\SceneOrganized\Newnorms\vadmat_scene{}_slidingnorm_{}s_v{}a{}d{}.csv'.format(targetscene, windowsize,v_n,a_n,d_n), index=False)